# シミュレーション計算実行用ノートブック

## 1. 今回の実験の説明

In [ ]:
DESCRIPTION = '''
mlflowを使ってシミュレーションプログラムの実行とその結果を管理する実験の例題1。DEM5のシミュレーションを行う。
'''
ISSUE_NO = ''
EXEC_NAME = 'DEM5'
RUN_SCRIPT = "calcDEM5"

PREV_RUNID = ''

MLFLOW_EXP_TYPE = "particleDEM"

dump_files = ["dump.DEM5", "dump.DEM5box"]

checking GPU devices
GPU status: no error
found 1 GPU
  Device 0: NVIDIA H100 80GB HBM3
NVIDIA H100 80GB HBM3 (compute capability 9)
  number of multiprocessor: 132
  number of cores / MP: 128
  global memory size: 3.1804 [GB]
  max threads per block: 1024
  max block per grid: 2147483647
  shared memory size: 48 [KB]


## 2. シミュレーションパラメータ

In [ ]:
import importlib, json
sim = importlib.import_module(f'scripts.{RUN_SCRIPT}')

## units used in this simulation are
## [g][cm][s]
sim_params = sim.default_prams

## 必要なら適宜修正する
sim_params["stepmax"] = 0.5


print(json.dumps(sim_params, indent=4, ensure_ascii=False))

{
    "R0": 0.5,
    "cell": [
        0.0,
        60.0,
        0.0,
        60.0,
        0.0,
        50.0
    ],
    "density": 7.874,
    "DEM_params": {
        "E": 21100000000.0,
        "mu": 0.4,
        "sigma": 0.29,
        "gamma": 0.10332,
        "mu_r": 0.1
    },
    "cutoff_block_factor": 0.9,
    "stepmax": 0.5,
    "intaval": 0.005,
    "initDeltaT": 8e-06,
    "ulim": 0.04,
    "param_g": 980.0
}


## 3. mlflow変数

In [3]:
import mlflow
import os
from time import strftime, gmtime
from utils.info_utils import MLflowEnvLogger

In [ ]:
ROOT = os.getenv("HOME")

## 1台構成の時
MLFLOW_TRACKING_URI = f"sqlite:///{ROOT}/mlruns/mlflow.db"
MLFLOW_STORAGE = f"file://{ROOT}/mlstorage"
### S3 bucket を指定する場合
#MLFLOW_STORAGE = "s3://my-mlflow-artifact-s3-bucket/mlstorage/"

## mlflow serverのIPを指定
#MLFLOW_TRACKING_URI = "http://<サーバのIP>:5000"


## github, backlogなどでチケット管理をしている場合はそのBASE URLを設定
ISSUE_BASE_URL = 'https://xxxxx/'

### dump fileの圧縮に使うコマンド（pixz があれば推奨）
ARCHIVE_COMMAND = "pixz"

###
### mlflow変数　自動設定
###
MYNAME = os.getenv("USER")
#GIT_INFO = gitutils.get_info()
RUN_NAME = EXEC_NAME + strftime("-%Y-%m-%d-%H-%M-%S", gmtime())

if ISSUE_NO != '':
    ISSUE_NAME = f'\n[{ISSUE_NO}]({ISSUE_BASE_URL}{ISSUE_NO})'
else:
    ISSUE_NAME = ''


## 4. シミュレーション実行

In [5]:
###
### mlflow処理開始
###
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow_exp = mlflow.get_experiment_by_name(MLFLOW_EXP_TYPE)
if mlflow_exp is None:
    mlflow_exp_id = mlflow.create_experiment(name=MLFLOW_EXP_TYPE, artifact_location=MLFLOW_STORAGE)
else:
    mlflow_exp_id = mlflow_exp.experiment_id

2026/02/22 04:18:00 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.schemas
2026/02/22 04:18:00 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.tables
2026/02/22 04:18:00 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.types
2026/02/22 04:18:00 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.constraints
2026/02/22 04:18:00 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.defaults
2026/02/22 04:18:00 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.comments
2026/02/22 04:18:00 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/02/22 04:18:00 INFO alembic.runtime.migration: Will assume non-transactional DDL.


In [ ]:
mlflow_run = mlflow.start_run(
    experiment_id=mlflow_exp_id,
    run_name=RUN_NAME,
    description=f'{DESCRIPTION}{ISSUE_NAME}',
    log_system_metrics=True)

print(f"Run ID: {mlflow_run.info.run_id}")

mlflow.set_tag("mlflow.user", MYNAME)
mlflow.set_tag("simulation", EXEC_NAME)
mlflow.set_tag("run_script", RUN_SCRIPT)
#mlflow.log_params({'git_commit': GIT_INFO['commit'], 'git_branch': GIT_INFO['branch']})
#mlflow.log_artifact('git.diff.txt', artifact_path='git_info')

2026/02/22 04:18:00 INFO mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics. Set logger level to DEBUG for more details.
2026/02/22 04:18:00 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.


Run ID: 75954256c0314bcca4fc816d9e2f9a8a


In [ ]:
##
## シミュレーション処理　シミュレーション初期化
##
sim.log_GPU_info(mlflow.set_tags)
mlflow.log_params(sim_params)
mlflow.set_tags(MLflowEnvLogger.log_all_env_tags())

if PREV_RUNID:
    prev_state_path = mlflow.artifacts.download_artifacts(
        run_id=PREV_RUNID,
        artifact_path="final_state",    # 取得したいアーティファクト内のパス
    )
    files = os.listdir(prev_state_path)
    sim.load_previous_state(f'{prev_state_path}/{files[0]}')
    os.system(f"rm -rf {prev_state_path}")
    mlflow.log_param("prev_run_id", PREV_RUNID)
else:
    sim.create_initial_state(sim_params, mlflow.log_params, dump_files[1])

N= 6121
N= 16921
particles object generated at: Sun Feb 22 04:18:01 2026
CUDA thread parameters: MPnum: 132, THnum1D: 1024, THnum1DX: 512, THnum2D: 32
CutoffBlock::setup
rmax: 0.51962
cell size is
0:60
0:60
0:50
60  0.51962 x 116
60  0.51962 x 116
50  0.51962 x 97
r0: 0.45
(SingleParticleBlock) blocknum changed: 116(0.51962) 116(0.51962) 97(0.51962) 
Total Number of Particles 16921 / total number of blocks 1305232 mean 0.012964
cuda grid for i-j pair table: 1305232 x 125
THnum2D2: 32
prange:0:16921, 6121:16921


In [ ]:
##
## シミュレーション処理　メインループ
##
archive = f'_{RUN_NAME}'
sim.run(sim_params, mlflow.log_metrics, archive, dump_files[0])

End Time:  0.5
0 output in TMPthread: 0 elapsed 0 sec
DeltaT(0):0:0:8e-06
Resize: 0 to 512 (pitch: 2048) 
DeltaT(+):8e-06:1:1.1314e-05
Resize: 0 to 512 (pitch: 2048) 
DeltaT(+):1.9314e-05:2:1.6e-05

DeltaT(+):3.5314e-05:3:2.2627e-05
50 100 150 200 (0.0050133452750742435)
output in TMPthread: 224 elapsed 0 sec
250 300 350 400 (0.01001400500535965)
450 output in TMPthread: 445 elapsed 0 sec
500 550 600 
DeltaT(+):0.014064:623:3.2e-05650 
(0.01502431184053421)
700 output in TMPthread: 654 elapsed 0 sec
750 800 (0.02001631259918213)
output in TMPthread: 810 elapsed 1 sec
850 900 950 (0.0250083114951849)
output in TMPthread: 966 elapsed 1 sec
1000 1050 Resize: 512 to 1024 (pitch: 4096) Resize: 512 to 1024 (pitch: 4096) Resize: 1024 to 1536 (pitch: 6144) Resize: 1024 to 1536 (pitch: 6144) 1100 Resize: 1536 to 2048 (pitch: 8192) (0.030000312253832817)
1150 Resize: 1536 to 2048 (pitch: 8192) output in TMPthread: 1122 elapsed 1 sec
Resize: 2048 to 2560 (pitch: 10240) Resize: 2048 to 2560 (pitch

Done.


In [ ]:
##
## シミュレーション処理　結果の登録・保存
##
for dump_file in dump_files:
    if dump_file:
        os.system(f"{ARCHIVE_COMMAND} {dump_file}")
        mlflow.log_artifact(f"{dump_file}.xz", artifact_path='dump')

mlflow.log_artifact(archive, artifact_path='final_state')


mlflow.end_run()

2026/02/22 04:18:32 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/02/22 04:18:32 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!


In [10]:
## 異常・中断時の mlflow.end_run()
run_info = mlflow.get_run(mlflow_run.info.run_id)
if run_info.info.lifecycle_stage == "active":
    mlflow.end_run(status='KILLED')
